In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd

# 👉 hardcode your file here
file_path = "/home/moshtasa/Research/phd-svd-recsys/Mobile/0405/0405_SVD/result/data/f_Events_60u_pos5_neg1_allapps_15.csv"

df = pd.read_csv(file_path)

# clean column names (important)
df.columns = df.columns.str.strip()

print("==== COLUMNS ====")
print(df.columns.tolist())

print("\n==== HEAD ====")
print(df.head())

print("\n==== INFO ====")
print(df.info())

==== COLUMNS ====
['uid', 'app_package', 'est_score', 'rank', 'category_g1', 'category_g2', 'categories_all']

==== HEAD ====
                uid                                  app_package  est_score  \
0  aO8q1gwOYrIriJ5M          com.bigshotgames.legendaryItemSuper   8.220789   
1  aO8q1gwOYrIriJ5M  com.hazelfunstudio.dream.garden.maker.story   8.191964   
2  aO8q1gwOYrIriJ5M            de.medando.bloodpressurecompanion   8.111129   
3  aO8q1gwOYrIriJ5M              com.apowersoft.backgrounderaser   8.037553   
4  aO8q1gwOYrIriJ5M                com.visiblebody.musclepremium   7.908565   

   rank   category_g1  category_g2 categories_all  
0     1  Role Playing          NaN   Role Playing  
1     2  Role Playing          NaN   Role Playing  
2     3       Medical          NaN        Medical  
3     4  Art & Design          NaN   Art & Design  
4     5       Medical          NaN        Medical  

==== INFO ====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10501665 entries, 0 t

In [4]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import defaultdict

# =========================
# PATHS
# =========================

DATA_DIR = Path("/home/moshtasa/Research/phd-svd-recsys/Mobile/0405/0405_SVD/result/data")

OUT_BASE = Path("/home/moshtasa/Research/phd-svd-recsys/Mobile/0405/0405_SVD/result/SVD_bar_figures")
OUT_BASE.mkdir(parents=True, exist_ok=True)

# =========================
# STORAGE
# =========================

results = defaultdict(lambda: defaultdict(dict))

# =========================
# UNSANITIZE CATEGORY
# =========================

def unsanitize_category(sanitized, known_categories=None):
    """Reverse filename sanitization back to actual category name.
    E.g. 'Travel___Navigation' -> 'Travel & Navigation'
    """
    if known_categories:
        for cat in known_categories:
            s = cat.replace(" ", "_")
            s = re.sub(r"[^0-9A-Za-z_]+", "_", s)
            if s == sanitized:
                return cat
    # Fallback
    s = re.sub(r'___', ' & ', sanitized)
    s = s.replace('_', ' ')
    return s

# =========================
# PARSE FILENAME
# =========================

def parse_filename(fname):
    if fname.startswith("ORIGINAL"):
        k = int(re.search(r'(\d+)', fname).group(1))
        return "ORIGINAL", "original", k

    match = re.match(r"f_(.*?)_(\d+u)_.*?_(\d+)recommendation\.csv", fname)
    if match:
        category_raw = match.group(1)
        n = match.group(2)
        k = int(match.group(3))
        return category_raw, n, k

    return None, None, None

# =========================
# METRIC
# =========================

def compute_avg(df, target_category):
    df['has_category'] = df['categories_all'].str.lower() == target_category.lower()
    user_counts = df.groupby('uid')['has_category'].sum()
    return user_counts.mean(), len(user_counts)

# =========================
# COLLECT KNOWN CATEGORIES FROM ORIGINAL
# =========================

known_categories = set()

for fname in os.listdir(DATA_DIR):
    if fname.startswith("ORIGINAL") and fname.endswith(".csv"):
        df_tmp = pd.read_csv(DATA_DIR / fname)
        for c in df_tmp['categories_all'].dropna().unique():
            c = str(c).strip()
            if c and c != "Unknown":
                known_categories.add(c)
        break  # One file is enough to get all categories

print(f"Known categories: {known_categories}")

# =========================
# READ NON-ORIGINAL
# =========================

all_n_values = set()

for fname in os.listdir(DATA_DIR):
    if not fname.endswith(".csv"):
        continue

    category_raw, n, k = parse_filename(fname)
    if category_raw is None or n == "original":
        continue

    category = unsanitize_category(category_raw, known_categories)

    df = pd.read_csv(DATA_DIR / fname)
    avg_val, num_users = compute_avg(df, category)

    results[category][k][n] = (avg_val, num_users)
    all_n_values.add(n)

# =========================
# ORIGINAL FILES
# =========================

original_data = defaultdict(dict)

for fname in os.listdir(DATA_DIR):
    if not fname.startswith("ORIGINAL"):
        continue

    _, _, k = parse_filename(fname)
    df = pd.read_csv(DATA_DIR / fname)

    # Get all categories except Unknown
    all_cats = set(df['categories_all'].dropna().str.strip().unique())
    all_cats.discard("Unknown")

    for cat in all_cats:
        avg_val, num_users = compute_avg(df.copy(), cat)
        original_data[cat][k] = (avg_val, num_users)

# =========================
# MERGE ORIGINAL
# =========================

for category in results.keys():
    for k in results[category].keys():
        if category in original_data and k in original_data[category]:
            results[category][k]['original'] = original_data[category][k]

# =========================
# COLORS (FINAL STYLE PALETTE)
# =========================

def generate_colors(n_list):
    palette = [
        "#4E79A7", "#59A14F", "#E15759", "#B07AA1",
        "#F28E2B", "#76B7B2", "#EDC948", "#9C755F",
        "#FF9DA7", "#8CD17D", "#86BCB6", "#BAB0AC"
    ]
    return {n: palette[i % len(palette)] for i, n in enumerate(n_list)}

# =========================
# PLOTTING
# =========================

for category, k_data in results.items():

    ks = sorted(k_data.keys())

    n_types = sorted(set(
        n for k in ks for n in k_data[k].keys()
    ), key=lambda x: (x != "original", int(x.replace("u", "")) if x != "original" else -1))

    color_map = generate_colors(n_types)

    x = np.arange(len(ks)) * 1.2
    width = 1.0 / len(n_types)

    fig, ax = plt.subplots(figsize=(18, 11))

    max_val = 0
    bar_containers = []

    for i, n in enumerate(n_types):
        vals = [k_data[k].get(n, (0, 0))[0] for k in ks]

        bars = ax.bar(
            x + (i - len(n_types)/2) * width + width/2,
            vals,
            width=width,
            color=color_map[n],
            edgecolor="black",
            linewidth=0.6,
            label=n
        )

        bar_containers.append((bars, vals))
        max_val = max(max_val, max(vals))

    # =========================
    # STYLE (UPDATED)
    # =========================

    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)

    ax.yaxis.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)

    ax.set_title(f"svd - {category}", fontsize=40, fontweight="bold", pad=20)
    ax.set_xlabel("k (recommendation size)", fontsize=38, labelpad=14)
    ax.set_ylabel("Avg count per user", fontsize=38, labelpad=14)

    ax.set_xticks(x)
    ax.set_xticklabels(ks, fontsize=34)
    ax.tick_params(axis='y', labelsize=34)

    ax.legend(fontsize=22, ncol=3)

    # =========================
    # BAR LABELS
    # =========================

    for bars, vals in bar_containers:
        for bar, val in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val + max_val * 0.015,
                f"{val:.2f}",
                ha="center",
                va="bottom",
                fontsize=22,
                rotation=90,
            )

    ax.set_ylim(top=max_val * 1.25)

    # =========================
    # SAVE FIGURE
    # =========================

    # Use filesystem-safe name for folder/file (replace & with _)
    safe_name = category.lower().replace(' ', '_').replace('&', 'and')

    genre_folder = OUT_BASE / f"svd_{safe_name}_analysis"
    genre_folder.mkdir(parents=True, exist_ok=True)

    fig_path = genre_folder / f"svd - {category}.png"

    plt.tight_layout()
    plt.savefig(fig_path, dpi=300)
    plt.close()

    # =========================
    # SAVE EXPLANATION TEXT
    # =========================

    explanation_path = genre_folder / f"svd - {category}.txt"

    with open(explanation_path, "w") as f:
        f.write(f"svd Analysis for Category: {category}\n\n")
        f.write("Metric: Average count of target category per user\n\n")

        for k in ks:
            f.write(f"k = {k}\n")
            for n in n_types:
                val, users = k_data[k].get(n, (0, 0))
                f.write(f"  {n}: {val:.4f} (users={users})\n")
            f.write("\n")

print("All svd figures and explanation files saved.")


Known categories: {'Books & Reference', 'Art & Design', 'Food & Drink', 'Puzzle', 'Dating', 'Tools', 'Finance', 'News & Magazines', 'Health & Fitness', 'Social', 'Word', 'Lifestyle', 'Travel & Local', 'Comics', 'Medical', 'Auto & Vehicles', 'Maps & Navigation', 'Shopping', 'Music & Audio', 'Racing', 'Parenting', 'Weather', 'Casino', 'Entertainment', 'Simulation', 'Role Playing', 'Action', 'Photography', 'Educational', 'Personalization', 'Beauty', 'Strategy', 'Trivia', 'Communication', 'Sports', 'Video Players & Editors', 'Productivity', 'Arcade', 'Music', 'Libraries & Demo', 'House & Home', 'Board', 'Adventure', 'Business', 'Events', 'Card', 'Education', 'Casual'}
All svd figures and explanation files saved.


### paramethers

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
test = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/Mobile/0405/0405_NMF/result/NMF_run/data/f_Shopping_30u_pos5_neg1_allapps_15recommendation.csv")
test.columns
test.head(10)
### bar analysis_NMF
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import defaultdict

# =========================
# PATHS
# =========================

DATA_DIR = Path("/home/moshtasa/Research/phd-svd-recsys/Mobile/0405/0405_NMF/result/NMF_run/data")

OUT_BASE = Path("/home/moshtasa/Research/phd-svd-recsys/Mobile/0405/0405_NMF/result/NMF_run/NMF_bar_figures")
OUT_BASE.mkdir(parents=True, exist_ok=True)

# =========================
# STORAGE
# =========================

results = defaultdict(lambda: defaultdict(dict))

# =========================
# UNSANITIZE CATEGORY
# =========================

def unsanitize_category(sanitized, known_categories=None):
    """Reverse filename sanitization back to actual category name.
    E.g. 'Travel___Navigation' -> 'Travel & Navigation'
    """
    if known_categories:
        for cat in known_categories:
            s = cat.replace(" ", "_")
            s = re.sub(r"[^0-9A-Za-z_]+", "_", s)
            if s == sanitized:
                return cat
    # Fallback
    s = re.sub(r'___', ' & ', sanitized)
    s = s.replace('_', ' ')
    return s

# =========================
# PARSE FILENAME
# =========================

def parse_filename(fname):
    if fname.startswith("ORIGINAL"):
        k = int(re.search(r'(\d+)', fname).group(1))
        return "ORIGINAL", "original", k

    match = re.match(r"f_(.*?)_(\d+u)_.*?_(\d+)recommendation\.csv", fname)
    if match:
        category_raw = match.group(1)
        n = match.group(2)
        k = int(match.group(3))
        return category_raw, n, k

    return None, None, None

# =========================
# METRIC
# =========================

def compute_avg(df, target_category):
    df['has_category'] = df['category'].str.lower() == target_category.lower()
    user_counts = df.groupby('uid')['has_category'].sum()
    return user_counts.mean(), len(user_counts)

# =========================
# COLLECT KNOWN CATEGORIES FROM ORIGINAL
# =========================

known_categories = set()

for fname in os.listdir(DATA_DIR):
    if fname.startswith("ORIGINAL") and fname.endswith(".csv"):
        df_tmp = pd.read_csv(DATA_DIR / fname)
        for c in df_tmp['category'].dropna().unique():
            c = str(c).strip()
            if c and c != "Unknown":
                known_categories.add(c)
        break  # One file is enough to get all categories

print(f"Known categories: {known_categories}")

# =========================
# READ NON-ORIGINAL
# =========================

all_n_values = set()

for fname in os.listdir(DATA_DIR):
    if not fname.endswith(".csv"):
        continue

    category_raw, n, k = parse_filename(fname)
    if category_raw is None or n == "original":
        continue

    category = unsanitize_category(category_raw, known_categories)

    df = pd.read_csv(DATA_DIR / fname)
    avg_val, num_users = compute_avg(df, category)

    results[category][k][n] = (avg_val, num_users)
    all_n_values.add(n)

# =========================
# ORIGINAL FILES
# =========================

original_data = defaultdict(dict)

for fname in os.listdir(DATA_DIR):
    if not fname.startswith("ORIGINAL"):
        continue

    _, _, k = parse_filename(fname)
    df = pd.read_csv(DATA_DIR / fname)

    # Get all categories except Unknown
    all_cats = set(df['category'].dropna().str.strip().unique())
    all_cats.discard("Unknown")

    for cat in all_cats:
        avg_val, num_users = compute_avg(df.copy(), cat)
        original_data[cat][k] = (avg_val, num_users)

# =========================
# MERGE ORIGINAL
# =========================

for category in results.keys():
    for k in results[category].keys():
        if category in original_data and k in original_data[category]:
            results[category][k]['original'] = original_data[category][k]

# =========================
# COLORS (FINAL STYLE PALETTE)
# =========================

def generate_colors(n_list):
    palette = [
        "#4E79A7", "#59A14F", "#E15759", "#B07AA1",
        "#F28E2B", "#76B7B2", "#EDC948", "#9C755F",
        "#FF9DA7", "#8CD17D", "#86BCB6", "#BAB0AC"
    ]
    return {n: palette[i % len(palette)] for i, n in enumerate(n_list)}

# =========================
# PLOTTING
# =========================

for category, k_data in results.items():

    ks = sorted(k_data.keys())

    n_types = sorted(set(
        n for k in ks for n in k_data[k].keys()
    ), key=lambda x: (x != "original", int(x.replace("u", "")) if x != "original" else -1))

    color_map = generate_colors(n_types)

    x = np.arange(len(ks)) * 1.2
    width = 1.0 / len(n_types)

    fig, ax = plt.subplots(figsize=(18, 11))

    max_val = 0
    bar_containers = []

    for i, n in enumerate(n_types):
        vals = [k_data[k].get(n, (0, 0))[0] for k in ks]

        bars = ax.bar(
            x + (i - len(n_types)/2) * width + width/2,
            vals,
            width=width,
            color=color_map[n],
            edgecolor="black",
            linewidth=0.6,
            label=n
        )

        bar_containers.append((bars, vals))
        max_val = max(max_val, max(vals))

    # =========================
    # STYLE (UPDATED)
    # =========================

    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)

    ax.yaxis.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)

    ax.set_title(f"NMF - {category}", fontsize=40, fontweight="bold", pad=20)
    ax.set_xlabel("k (recommendation size)", fontsize=38, labelpad=14)
    ax.set_ylabel("Avg count per user", fontsize=38, labelpad=14)

    ax.set_xticks(x)
    ax.set_xticklabels(ks, fontsize=34)
    ax.tick_params(axis='y', labelsize=34)

    ax.legend(fontsize=22, ncol=3)

    # =========================
    # BAR LABELS
    # =========================

    for bars, vals in bar_containers:
        for bar, val in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val + max_val * 0.015,
                f"{val:.2f}",
                ha="center",
                va="bottom",
                fontsize=22,
                rotation=90,
            )

    ax.set_ylim(top=max_val * 1.25)

    # =========================
    # SAVE FIGURE
    # =========================

    # Use filesystem-safe name for folder/file (replace & with _)
    safe_name = category.lower().replace(' ', '_').replace('&', 'and')

    genre_folder = OUT_BASE / f"nmf_{safe_name}_analysis"
    genre_folder.mkdir(parents=True, exist_ok=True)

    fig_path = genre_folder / f"NMF - {category}.png"

    plt.tight_layout()
    plt.savefig(fig_path, dpi=300)
    plt.close()

    # =========================
    # SAVE EXPLANATION TEXT
    # =========================

    explanation_path = genre_folder / f"NMF - {category}.txt"

    with open(explanation_path, "w") as f:
        f.write(f"NMF Analysis for Category: {category}\n\n")
        f.write("Metric: Average count of target category per user\n\n")

        for k in ks:
            f.write(f"k = {k}\n")
            for n in n_types:
                val, users = k_data[k].get(n, (0, 0))
                f.write(f"  {n}: {val:.4f} (users={users})\n")
            f.write("\n")

print("All NMF figures and explanation files saved.")

### param
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import defaultdict

# ── Paths ──
BASE = "/home/moshtasa/Research/phd-svd-recsys/Mobile/0405/0405_NMF/result/NMF_run"
BI_DIR = os.path.join(BASE, "bi")
MU_DIR = os.path.join(BASE, "mio")
MERGED_DIR = os.path.join(BASE, "merged")
FIG_DIR = os.path.join(BASE, "NMF_details_figures")

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ── Unsanitize ──

def unsanitize_category(sanitized, known_categories=None):
    """Reverse filename sanitization back to actual category name.
    E.g. 'Travel___Navigation' -> 'Travel & Navigation'
    """
    if known_categories:
        for cat in known_categories:
            s = cat.replace(" ", "_")
            s = re.sub(r"[^0-9A-Za-z_]+", "_", s)
            if s == sanitized:
                return cat
    # Fallback
    s = re.sub(r'___', ' & ', sanitized)
    s = s.replace('_', ' ')
    return s

# ── Step 1: Discover all bi files and parse (category, n) from filenames ──
BI_PATTERN = re.compile(r"^f_(.+?)_(\d+)u_pos5_neg1_allapps_bi_per_category\.csv$")
MU_PATTERN = re.compile(r"^f_(.+?)_(\d+)u_pos5_neg1_allapps_mu_per_category\.csv$")

# Collect all (category_key, n) pairs from bi folder
bi_files = {}
for fname in os.listdir(BI_DIR):
    m = BI_PATTERN.match(fname)
    if m:
        category_key = m.group(1)
        n = int(m.group(2))
        bi_files[(category_key, n)] = fname

# ── Collect known categories from ORIGINAL bi file ──
known_categories = set()
orig_bi_path = os.path.join(BI_DIR, "ORIGINAL_bi_per_category.csv")
if os.path.exists(orig_bi_path):
    df_tmp = pd.read_csv(orig_bi_path)
    for c in df_tmp['category'].dropna().unique():
        c = str(c).strip()
        if c and c != "Unknown":
            known_categories.add(c)

print(f"Known categories: {known_categories}")

# ── Step 2: Merge bi + mu files and save ──
merged_data = {}

for (category_key, n), bi_fname in sorted(bi_files.items()):
    mu_fname = bi_fname.replace("_bi_", "_mu_").replace("/bi/", "/mio/")
    bi_path = os.path.join(BI_DIR, bi_fname)
    mu_path = os.path.join(MU_DIR, mu_fname)

    if not os.path.exists(mu_path):
        print(f"WARNING: No matching mu file for {bi_fname}, skipping.")
        continue

    df_bi = pd.read_csv(bi_path)
    df_mu = pd.read_csv(mu_path)

    # Exclude Unknown
    df_bi = df_bi[df_bi["category"] != "Unknown"]
    df_mu = df_mu[df_mu["category"] != "Unknown"]

    # Merge on category (and is_target if present in both)
    merge_cols = ["category"]
    if "is_target" in df_bi.columns and "is_target" in df_mu.columns:
        merge_cols.append("is_target")

    df_merged = pd.merge(df_mu, df_bi, on=merge_cols, how="outer")

    # Save merged file
    merged_fname = bi_fname.replace("_bi_", "_merged_")
    merged_path = os.path.join(MERGED_DIR, merged_fname)
    df_merged.to_csv(merged_path, index=False)

    merged_data[(category_key, n)] = df_merged

# Also merge the ORIGINAL files if they exist
orig_bi = os.path.join(BI_DIR, "ORIGINAL_bi_per_category.csv")
orig_mu = os.path.join(MU_DIR, "ORIGINAL_mu_per_category.csv")
if os.path.exists(orig_bi) and os.path.exists(orig_mu):
    df_obi = pd.read_csv(orig_bi)
    df_omu = pd.read_csv(orig_mu)
    df_obi = df_obi[df_obi["category"] != "Unknown"]
    df_omu = df_omu[df_omu["category"] != "Unknown"]
    merge_cols = ["category"]
    if "is_target" in df_obi.columns and "is_target" in df_omu.columns:
        merge_cols.append("is_target")
    df_orig_merged = pd.merge(df_omu, df_obi, on=merge_cols, how="outer")
    df_orig_merged.to_csv(os.path.join(MERGED_DIR, "ORIGINAL_merged_per_category.csv"), index=False)

print(f"Merged {len(merged_data)} file pairs into {MERGED_DIR}")

# ── Step 3: Create one figure per target category ──
category_groups = defaultdict(dict)
for (category_key, n), df in merged_data.items():
    category_groups[category_key][n] = df

# Style
plt.style.use("seaborn-v0_8-pastel")
TARGET_COLOR = "#5B3A29"
TARGET_LINEWIDTH = 3.5
NON_TARGET_LINEWIDTH = 1.5

PASTEL_COLORS = [
    "#AEC6CF", "#FFD1DC", "#B5EAD7", "#FFDAC1", "#C3B1E1",
    "#F3E5AB", "#FFB7B2", "#B0E0E6", "#D5AAFF", "#FFFACD",
    "#E0BBE4", "#957DAD", "#D4E157", "#80CBC4", "#F48FB1",
]

for category_key, n_dict in sorted(category_groups.items()):
    sorted_ns = sorted(n_dict.keys())

    # Determine the target category name from the is_target column
    sample_df = n_dict[sorted_ns[0]]
    target_rows = sample_df[sample_df["is_target"] == "target"]
    if target_rows.empty:
        print(f"WARNING: No target category found in {category_key} files, skipping figure.")
        continue
    target_category_name = target_rows["category"].iloc[0]

    # Collect all category names across all n values (excluding Unknown)
    all_categories = set()
    for n in sorted_ns:
        cats = n_dict[n]["category"].tolist()
        for c in cats:
            if str(c).strip() != "Unknown":
                all_categories.add(c)
    all_categories = sorted(all_categories)

    # Build data: for each category, collect (n, avg_bi + global_mean_mu)
    category_series = {}
    for cat_name in all_categories:
        points = []
        for n in sorted_ns:
            df = n_dict[n]
            row = df[df["category"] == cat_name]
            if not row.empty:
                val = row["avg_bi"].values[0] + row["global_mean_mu"].values[0]
                points.append((n, val))
        category_series[cat_name] = points

    non_target_categories = [c for c in all_categories if c != target_category_name]

    color_map = {}
    for i, c in enumerate(non_target_categories):
        color_map[c] = PASTEL_COLORS[i % len(PASTEL_COLORS)]
    color_map[target_category_name] = TARGET_COLOR

    positions = list(range(len(sorted_ns)))
    n_to_pos = {n: p for n, p in zip(sorted_ns, positions)}

    fig, ax = plt.subplots(figsize=(14, 8))

    # Plot non-target categories
    for cat_name in non_target_categories:
        pts = category_series[cat_name]
        if pts:
            xs = [n_to_pos[x] for x, _ in pts]
            ys = [y for _, y in pts]
            ax.plot(xs, ys, marker="o", color=color_map[cat_name],
                    linewidth=NON_TARGET_LINEWIDTH, markersize=9,
                    label=cat_name, alpha=0.8)

    # Plot target category on top
    pts = category_series[target_category_name]
    if pts:
        xs = [n_to_pos[x] for x, _ in pts]
        ys = [y for _, y in pts]
        ax.plot(xs, ys, marker="o", color=TARGET_COLOR,
                linewidth=TARGET_LINEWIDTH, markersize=7,
                label=f"{target_category_name} (target)", zorder=10)

    pretty_category = unsanitize_category(category_key, known_categories)
    ax.tick_params(axis='y', labelsize=18)
    ax.tick_params(axis='x', labelsize=24)
    ax.set_title(f"NMF - {pretty_category}", fontsize=28, fontweight='bold')
    ax.set_xlabel("Number of Fictitious Users (n)", fontsize=25)
    ax.set_ylabel(r"$\mu$ + $b_i$", fontsize=30, fontweight='bold', color='black')
    ax.set_xticks(positions)
    ax.set_xticklabels([str(n) for n in sorted_ns], rotation=45)
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    safe_name = category_key.replace("&", "and")
    fig_path = os.path.join(FIG_DIR, f"{safe_name}_NMF_parameters.png")
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {fig_path}")

print("Done!")



KeyError: 'categories_all'